# Consulta do dia — eventos `transaction-analyzed` (Kafka)

Após cada `POST /analyze`, a API:
1. Responde o score (síncrono)
2. Publica o evento no Kafka **de forma assíncrona**
3. Espelha o evento no Mongo (`analyzed_events`) para consulta

**Payload:** CPF + apenas `card_last4` (não o cartão completo).

URLs: API `http://localhost:8080` (host) ou `http://api:8080` (dentro do Compose).

In [ ]:
import json
from datetime import date, datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd
import requests

# Dentro do container Jupyter use api:8080; no host use localhost:8080
API_BASES = ["http://api:8080", "http://localhost:8080"]
DAY = "today"  # ou "2026-07-26"


def fetch_day(day: str = "today") -> dict:
    last_err = None
    for base in API_BASES:
        try:
            r = requests.get(
                f"{base}/api/v1/events/analyzed",
                params={"date": day},
                timeout=8,
            )
            r.raise_for_status()
            print(f"OK via {base}")
            return r.json()
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Não foi possível consultar a API: {last_err}")


payload = fetch_day(DAY)
print(json.dumps({k: payload[k] for k in ("event_day", "total", "frauds", "legit", "kafka_topic")}, indent=2))

In [ ]:
events = payload.get("events") or []
df = pd.DataFrame(events)
if df.empty:
    print("Sem eventos neste dia. Gere análises:")
    print("  curl -X POST http://localhost:8080/api/v1/transactions/analyze ...")
else:
    cols = [
        c
        for c in [
            "occurred_at",
            "transaction_id",
            "cpf",
            "card_last4",
            "amount",
            "fraud_score",
            "is_fraud",
            "risk_level",
            "merchant_category",
        ]
        if c in df.columns
    ]
    display(df[cols].head(30))

## Gráficos do dia

In [ ]:
if df.empty:
    print("Nada para plotar.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Pizza fraude vs legítimo
    fraud_counts = df["is_fraud"].fillna(False).astype(bool).value_counts()
    labels = ["Fraude" if v else "Legítimo" for v in fraud_counts.index]
    axes[0].pie(
        fraud_counts.values,
        labels=labels,
        autopct="%1.1f%%",
        colors=["#ef4444", "#22c55e"],
        startangle=90,
    )
    axes[0].set_title(f"Fraude × Legítimo — {payload.get('event_day')}")

    # Histograma de scores
    scores = pd.to_numeric(df["fraud_score"], errors="coerce").dropna()
    axes[1].hist(scores, bins=12, color="#38bdf8", edgecolor="white")
    axes[1].axvline(0.74, color="#f59e0b", linestyle="--", label="limiar 0,74")
    axes[1].set_title("Distribuição de fraud_score")
    axes[1].set_xlabel("score")
    axes[1].set_ylabel("qtd")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    if "merchant_category" in df.columns:
        fig2, ax2 = plt.subplots(figsize=(10, 4))
        by_cat = (
            df.assign(is_fraud=df["is_fraud"].fillna(False).astype(bool))
            .groupby("merchant_category")["is_fraud"]
            .agg(["count", "sum"])
            .rename(columns={"count": "total", "sum": "fraudes"})
            .sort_values("total", ascending=False)
            .head(8)
        )
        by_cat.plot(kind="bar", ax=ax2, color=["#64748b", "#ef4444"])
        ax2.set_title("Volume e fraudes por categoria")
        ax2.set_xlabel("")
        ax2.tick_params(axis="x", rotation=30)
        plt.tight_layout()
        plt.show()

## (Opcional) Conferir tópico Kafka no broker

No terminal do host (não precisa rodar no notebook):

```bash
docker exec kafka kafka-console-consumer \
  --bootstrap-server localhost:9092 \
  --topic transaction-analyzed \
  --from-beginning --max-messages 5
```